In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "tiendung"
CODAPATH = Path("/kaggle/working/codapath")
if (CODAPATH / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(CODAPATH), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "pull", "--ff-only", "origin", REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f"{CODAPATH} exists but is not a Git repository")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(CODAPATH)])
actual_branch = subprocess.check_output(["git", "-C", str(CODAPATH), "branch", "--show-current"], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print("repo:", CODAPATH, "| branch:", actual_branch)

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# ---- EDIT THIS CELL ----
# pathmnist | histoset | skintissue
DATASET = "pathmnist"

# random | coreset | typiclust | activeft | badge | entropy | margin
# codapath | scalpel | nucleus_al | uncertainty_herding | tcm | dropquery | refine
SAMPLER_NAME = "nucleus_al"
SEED = 42

# nucleus_al controlled variants:
# 1) {"cell_source": "crop_dino", "uncertainty_mode": "cell_margin"}
# 2) {"cell_source": "cellvit_embedding", "uncertainty_mode": "disagreement"}
# 3) {"cell_source": "cellvit_embedding", "uncertainty_mode": "fusion_concat"}
# 4) {"cell_source": "cellvit_embedding", "uncertainty_mode": "fusion_add"}
SAMPLER_OVERRIDES = {"cell_source": "cellvit_embedding", "uncertainty_mode": "disagreement"}

# Leave RUN_NAME=None to derive a collision-safe name from the variant.
RUN_NAME = None

# Writable DINO cache. Replace with a mounted cache only when its manifest matches.
FEATURE_DIR = "/kaggle/working/features"
# Extraction output must first be saved as a Kaggle Dataset and attached here.
# Example: /kaggle/input/pathmnist-nucleus-cache/nucleus_features
NUCLEUS_FEATURE_DIR = "/kaggle/input/EDIT_NUCLEUS_CACHE_SLUG/nucleus_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public: no placeholder login/token is required.
# Enable Kaggle Internet, or set the model path to a mounted local snapshot.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from run import main

In [ ]:
PATHMNIST_PATH  = "/kaggle/input/nckh2026/pathmnist_224.npz"
HISTOSET_PATH   = "/kaggle/input/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert DATASET in DATA_DICT, DATASET
data_path = Path(DATA_DICT[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert config["cumulative_budget"] == [25, 50, 75, 100, 125, 150, 175, 200]
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"

training_cfg = config.get("training", {})
dataset_info = config["datasets"][DATASET]
sampler_cfg = dict(config.get("samplers", {}).get(SAMPLER_NAME, {}))
sampler_cfg.update(SAMPLER_OVERRIDES)
if SAMPLER_NAME == "nucleus_al":
    nucleus_manifest = (
        Path(NUCLEUS_FEATURE_DIR) / f"{DATASET}_seed{SEED}" / "manifest.json"
    )
    assert nucleus_manifest.is_file(), (
        f"Missing nucleus cache: {nucleus_manifest}. Attach the extraction notebook output "
        "and edit NUCLEUS_FEATURE_DIR in the configuration cell."
    )
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"[{SAMPLER_NAME}] sampler_cfg =", sampler_cfg)
print("data:", data_path, "| output:", OUTPUT_DIR)

In [ ]:
main(
    data_path=str(data_path),
    sampler_name=SAMPLER_NAME,
    num_classes=dataset_info["num_classes"],
    cumulative_budget=config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    device=torch.device(config["device"]),
    random_seed=SEED,
    save_dir=str(Path(OUTPUT_DIR) / DATASET),
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
    nucleus_cache_dir=NUCLEUS_FEATURE_DIR,
    run_name=RUN_NAME,
)